# behaviors

> The client runtime: delegated event rules that survive htmx swaps

In [ ]:
#| default_exp behaviors

The registry stores a component's behaviors as records: a selector, an event, a handler body. Here we turn those into running code in the browser. Wiring a listener onto each element as it appears would work for a static page, but in a FastHTML app elements arrive continuously as htmx swaps in fragments, so per-element wiring would mean re-initialising after every swap.

Instead we delegate: every event becomes one document-level listener which, when its event fires anywhere on the page, checks whether the target sits inside an element matching the rule's selector. Matching happens at event time against the live DOM, so elements swapped in later behave the same as those in the original page, with no per-element state and no mutation observer. The rule set is plain data, emitted fresh on each page render, which also lets a hyperelement be redefined in a live notebook.

In [ ]:
from fastcore.utils import *
from answerui.registry import Registry

In [ ]:
from answerui.registry import reg, hypelt
from fastcore.test import *

## The runtime

The script below is the whole runtime. Rules arrive as a list of records, each with a selector, the event names to listen on, a capture flag, and a handler function. We group them by event and phase, so each pair gets one listener however many rules share it. On an event, each rule in the group looks for the nearest ancestor of the target (the target included) that matches its selector, and calls the handler with it.

An `init` rule runs when an element arrives in the document. It registers through `htmx.onLoad`, htmx's hook for arriving content, which calls back with the root of newly swapped content, and with the body for the initial page. The rule matches the root itself and anything inside it. Walking up from the root instead would run a handler for an ancestor that was already on the page, as though it had just arrived. The handler's `e` is an `Event` of type `init` that was not dispatched. `htmx.onLoad` needs htmx loaded before this script, which FastHTML's default headers and `demo` both do.

FastHTML sends the headers only with a full page, never in the fragment it returns to an htmx request (and htmx discards the `<head>` of any full page it swaps anyway), so the script runs only once per document.

Handlers are async functions, so a failure in one rejects its own promise and leaves the other rules on that event running. Listening in the capture phase is opt-in, for events like `toggle` and `scroll` that don't bubble. The script is installed once per document: a regenerated script reaches a fresh page or preview, and a second copy injected into the same document would stack its listeners on the first.

In [ ]:
_runtime = r"""(() => {
%s
const rules = [%s];
const by = {};
rules.forEach(r => r.evs.forEach(ev => (by[ev + '|' + (r.cap ? 1 : 0)] ??= []).push(r)));
for (const k in by) {
  const [ev, cap] = k.split('|');
  document.addEventListener(ev, e => {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      const el = t.closest(r.sel);
      if (el) r.fn(el, e);
    }
  }, {capture: cap === '1'});
}
const init = rules.filter(r => r.init);
if (init.length) htmx.onLoad(root => {
  const e = new Event('init');
  for (const r of init) {
    if (root.matches(r.sel)) r.fn(root, e);
    root.querySelectorAll(r.sel).forEach(el => r.fn(el, e));
  }
});
})();"""

## Entries

Each registry record becomes one object literal in that list. Two event names belong to the library: `init`, when an element arrives in the document, and `cleanup`, before htmx removes one. Any other name is a DOM event, and a leading `:` is shorthand for an htmx event, so `':after-request'` listens for `htmx:after-request`.

htmx spells its events three ways. htmx 2 dispatches `htmx:afterSwap` and the alias `htmx:after-swap`. htmx 4 renamed its events to the form `htmx:after:swap`, and FastHTML's `htmx4=True` sets `metaCharacter` to `-`, which htmx 4 applies to every colon, giving `htmx-after-swap`. `_event` reduces any spelling to one canonical name, `htmx:` plus the hyphenated form, and `_spellings` lists every DOM name a canonical event fires under. An entry listens on all of them, and only one fires on a given page. The library's two names are handled apart. `init` is not an event name: it registers through `htmx.onLoad`, which knows its own version's arrival event. `cleanup` is a rename as well as a respelling, `htmx:beforeCleanupElement` in 2 and `htmx:before:cleanup` in 4, so `_spellings` lists its names outright. Under htmx 4, `cleanup` reaches only elements htmx initialised, those carrying `hx-` attributes. Each distinct handler body becomes one async function taking `el` and `e`, declared before the list, and an entry names its function:

In [ ]:
_cleanup = ['htmx:before-cleanup-element', 'htmx:before:cleanup', 'htmx-before-cleanup']

def _event(ev):
    "The canonical event name: a leading `:` is the htmx shorthand, and every spelling of an htmx event becomes `htmx:` plus its hyphenated name"
    if ev.startswith(':'): ev = 'htmx'+ev
    if not ev.startswith(('htmx:', 'htmx-')): return ev
    return 'htmx:' + re.sub(r'(?<=[a-z])(?=[A-Z])', '-', ev[5:]).lower().replace(':', '-')

def _spellings(ev):
    "The DOM event names a canonical event fires under: htmx 2's hyphenated alias, htmx 4's colon form, and htmx 4's form under FastHTML's `metaCharacter`"
    if ev == 'init': return []
    if ev == 'cleanup': return _cleanup
    if not ev.startswith('htmx:'): return [ev]
    name = ev[5:]
    return uniqueify([ev, 'htmx:'+name.replace('-', ':'), 'htmx-'+name])

def _entry(r, fn):
    ev = _event(r['ev'])
    return f'{{sel: {dumps(r["sel"])}, evs: {dumps(_spellings(ev))}, cap: {dumps(r["cap"])}, init: {dumps(ev=="init")}, fn: {fn}}}'


Every spelling of an htmx event is one identity, and an entry listens on each DOM name the event fires under:

In [ ]:
for ev in [':after-swap', 'htmx:afterSwap', 'htmx:after:swap', 'htmx-after-swap']: test_eq(_event(ev), 'htmx:after-swap')
_spellings('htmx:after-swap')

['htmx:after-swap', 'htmx:after:swap', 'htmx-after-swap']

## Building the script

A behavior's identity is its selector, its event, and its capture flag, with the event name made canonical first. Declaring the same identity again replaces the earlier handler: the last declaration in registry order ships and the others are dropped. Re-registering a rule keeps its position, so re-running a component's definition never outranks an application's later override. A replacement also keeps the slot of the handler it replaced, so dispatch order within an event follows first declarations. Only an exact match counts. Two selectors that happen to pick the same elements are two behaviors, and both run, since the registry does not read CSS.

`behaviors` is the script's input, the effective handler for each identity with the key of the rule that declared it, and `build_js` fills the runtime with it:


In [ ]:
@patch
def behaviors(self:Registry):
    "The effective behavior per (selector, event, capture): the last declared in registry order, with the key of the rule that declared it"
    res = {}
    for k,r in self.rules.items():
        for b in r.on: res[(b['sel'], _event(b['ev']), b['cap'])] = dict(b, ev=_event(b['ev']), rule=k)
    return res

@patch
def build_js(self:Registry):
    "The behavior script: one function per distinct handler, one entry per effective behavior"
    bs = self.behaviors().values()
    fns = {js: f'f{i}' for i,js in enumerate(uniqueify([b['js'] for b in bs]))}
    decls = '\n'.join(f'async function {nm}(el, e) {{ {js} }}' for js,nm in fns.items())
    return _runtime % (decls, ',\n'.join(_entry(b, fns[b['js']]) for b in bs))


As examples, let's make a notice that removes itself after a delay (or when a child marked `dismiss` is clicked), and a menu that focuses its first item when its popover opens. The autodismiss is an `init` rule. htmx fires the arrival event for the initial page and for every swapped fragment, but never for content already on the page, so swapping something inside a notice doesn't restart it. Since `toggle` doesn't bubble, the menu rule uses the capture phase. Capture makes the event observable, not exclusive to the matched element: a `details` toggled inside the menu would reach the handler too, so it checks that the target is the menu itself:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
reg.rule('dropdown', sel='[popover][dropdown]', on={('&', 'toggle', True): r"if (e.target === el && e.newState === 'open') el.querySelector('a, button')?.focus()"})
print('\n'.join(l for l in reg.build_js().splitlines() if 'notice-box' in l or '[dropdown]' in l))

async function f1(el, e) { el.closest('notice-box').remove() }
const rules = [{sel: ":is(notice-box)[autodismiss]", evs: [], cap: false, init: true, fn: f0},
{sel: ":is(notice-box) [dismiss]", evs: ["click"], cap: false, init: false, fn: f1},
{sel: ":is([popover][dropdown])", evs: ["toggle"], cap: true, init: false, fn: f2}];


The `init` rule has `init` set and no event names, since it registers through `htmx.onLoad`. The `toggle` rule carries `cap`. The click rule is plain. The browser does the grouping into listeners, so the script itself is only the functions and the rule list.

In [ ]:
s = reg.build_js()
assert 'init: true' in s and 'htmx.onLoad' in s and 'cap: true' in s

A rule often gives one handler to several events. Here a form is marked `dirty` on `input` and again on `change`, since a select fires only the second. The script declares that body once, and both entries name the same function:

In [ ]:
_dirty = r"el.toggleAttribute('dirty', [...el.elements].some(c => c.value !== c.defaultValue))"
reg.rule('dirty', sel='form', on={('&', 'input'): _dirty, ('&', 'change'): _dirty})
s = reg.build_js()
test_eq(s.count(_dirty), 1)
test_eq(s.count('":is(form)"'), 2)

An application that wants a confirmation before a notice goes registers the same selector and event through the component. `&` expands exactly as it did in the definition, so the identity matches and the confirming handler replaces the original. The effective behaviors say which rule each handler came from:

In [ ]:
NoticeBox.rule('confirm-dismiss', on={('& [dismiss]', 'click'): r"if (confirm('Dismiss this notice?')) el.closest('notice-box').remove()"})
{k: b['rule'] for k,b in reg.behaviors().items()}

{(':is(notice-box)[autodismiss]', 'init', False): 'NoticeBox.base',
 (':is(notice-box) [dismiss]', 'click', False): 'NoticeBox.confirm-dismiss',
 (':is([popover][dropdown])', 'toggle', True): 'dropdown',
 (':is(form)', 'input', False): 'dirty',
 (':is(form)', 'change', False): 'dirty'}

The script carries one click handler for the dismiss button, the confirming one. `NoticeBox.behaviors` still lists both declarations, because it reports what was declared on the component; `reg.behaviors` reports what ships:

In [ ]:
s = reg.build_js()
test_eq(s.count('":is(notice-box) [dismiss]"'), 1)
assert 'confirm(' in s
NoticeBox.behaviors

[{'sel': ':is(notice-box)[autodismiss]',
  'ev': 'init',
  'js': "await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
  'cap': False},
 {'sel': ':is(notice-box) [dismiss]',
  'ev': 'click',
  'js': "el.closest('notice-box').remove()",
  'cap': False},
 {'sel': ':is(notice-box) [dismiss]',
  'ev': 'click',
  'js': "if (confirm('Dismiss this notice?')) el.closest('notice-box').remove()",
  'cap': False}]

Re-registering the component's definition does not displace the application's override:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
test_eq(reg.behaviors()[(':is(notice-box) [dismiss]', 'click', False)]['rule'], 'NoticeBox.confirm-dismiss')
[k for k in reg.behaviors() if k[1]=='init']

[(':is(notice-box)[autodismiss]', 'init', False)]

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()